# **Drug Repurposing Pathfinding Benchmark Analysis**

This notebook analyzes the results of the graph pathfinding benchmark for drug–disease mechanism discovery.

The benchmark experiments are executed using `benchmark_runner.py`, which runs multiple pathfinding algorithms on curated drug–disease pathways from the PrimeKG knowledge graph. The resulting predictions and evaluation metrics are saved to disk and loaded here for analysis and visualization.

The goal of this notebook is to compare algorithm performance in terms of biological plausibility, pathway reconstruction accuracy, and structural properties of predicted paths.

<br>

---

## **Evaluation Metrics**

The following metrics are used to evaluate predicted pathways against curated ground-truth mechanisms:

**Precision:** Fraction of predicted nodes that appear in the ground truth pathway.

**Recall:** Fraction of ground truth nodes recovered.

**F1 Score:** Harmonic mean of precision and recall.

**Path Length Accuracy:** Agreement between predicted pathway length and ground truth pathway length.

**Hub Node Ratio:** Fraction of nodes in the predicted pathway that belong to the highest-degree nodes in the graph (top percentile). Lower values indicate more biologically plausible pathways.

**MRR (Mean Reciprocal Rank):** Measures how highly the correct nodes are ranked among candidate predictions.

**Edit Distance:** Measures similarity between predicted and ground truth node sequences.

<br>

---

## **Algorithms Evaluated**

The benchmark compares multiple graph search strategies:

**Dijkstra:** Baseline shortest-path algorithm minimizing hop distance.

**Hub-Penalized Shortest Path:** Applies penalties to high-degree nodes to discourage hub-dominated pathways.

**PageRank-Inverse Weighted Search:** Uses PageRank scores to downweight highly central nodes.

**Semantic Bridging:** Incorporates embedding similarity to discover semantically related biological entities.

**Bidirectional Search:** Simultaneously expands search from drug and disease nodes to improve efficiency.

**Relation-Weighted Bidirectional Search:** Extends bidirectional search by incorporating biological relation weighting.

**K-Shortest Biological Paths:** Returns multiple candidate pathways and selects biologically plausible routes.

<br>




This notebook focuses on interpreting the benchmark results through summary statistics, visualizations, and case study pathway analyses.

---

## Setup

Load benchmark results, predictions, and ground truth pathways from disk for analysis.

---

In [ ]:
# Load benchmark results
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
from difflib import SequenceMatcher
import networkx as nx

# Load results (go up one directory to parent, then into results/)
results_dir = Path('..') / 'results'
data_dir = Path('..') / 'data'

predictions = pd.read_csv(results_dir / 'all_predictions.csv')
results = pd.read_csv(results_dir / 'all_results.csv')
summary = pd.read_csv(results_dir / 'summary_by_algorithm.csv')
ground_truth_nodes = pd.read_csv(data_dir / 'benchmark_pathways_nodes.csv')
ground_truth_edges = pd.read_csv(data_dir / 'benchmark_pathways_edges.csv')

print(f"✓ Loaded {len(predictions)} predictions")
print(f"✓ Loaded {len(ground_truth_nodes)} ground truth nodes")
print(f"✓ Algorithms: {', '.join(results['algorithm'].unique())}")

---
## **1. Algorithm Perfomance Analysis**
---

### 📊 **Metric Summary**

We evaluated **8 pathfinding algorithms** on **150 drug-disease pathways** across 7 key metrics. The table below shows mean performance, sorted by F1 score (our primary metric for node accuracy).

**Color coding:** 🟢 Green = better performance | 🔴 Red = worse performance  
**Note:** For Edit Distance and Hub Node Ratio, lower values are better.

In [ ]:
# ============================================
#  SUMMARY TABLE 
# ============================================
summary_sorted = summary.sort_values('f1_score', ascending=False).copy()

styled_summary = summary_sorted.style\
    .format({
        'precision': '{:.4f}',
        'recall': '{:.4f}',
        'f1_score': '{:.4f}',
        'edit_distance': '{:.4f}',
        'mrr': '{:.4f}',
        'hub_node_ratio': '{:.4f}',
        'path_length_accuracy': '{:.4f}',
        'n_pathways': '{:.0f}'
    })\
    .set_properties(**{
        'text-align': 'center',
        'font-weight': 'normal'
    })

display(styled_summary)


In [ ]:
# Generate dynamic rankings
from IPython.display import Markdown, display

# Get top 3
top3 = summary_sorted.head(3)

medals = ['🥇', '🥈', '🥉']
places = ['1st Place', '2nd Place', '3rd Place']

md_text = "### **🏆 Top 3 Performers by F1 Score**\n\n"

for i, (idx, row) in enumerate(top3.iterrows()):
    md_text += f"**{medals[i]} {places[i]}: {row['algorithm']}** (F1: {row['f1_score']:.4f})  \n"
    md_text += f"ED: {row['edit_distance']:.4f} | Precision: {row['precision']:.4f} | Recall: {row['recall']:.4f}\n\n"

md_text += f"**Key Takeaway:** {top3.iloc[0]['algorithm']} achieved the highest F1 score ({top3.iloc[0]['f1_score']:.4f}), "
md_text += f"outperforming the baseline by {(top3.iloc[0]['f1_score'] - summary_sorted[summary_sorted['algorithm'] == 'Dijkstra']['f1_score'].values[0]):.4f} points."

display(Markdown(md_text))

In [ ]:
# ============================================
#  HEATMAP
# ============================================

LOWER_IS_BETTER = {'hub_node_ratio', 'edit_distance'}

# Prepare data
df = summary_sorted.copy()

# Set algorithm as index if it's a column
if 'algorithm' in df.columns:
    df = df.set_index('algorithm')

# Drop n_pathways if present
if 'n_pathways' in df.columns:
    df = df.drop(columns=['n_pathways'])

# Normalize data for coloring (0-1 scale, higher = better)
normalized = df.copy()
for col in df.columns:
    if col in LOWER_IS_BETTER:
        normalized[col] = 1 - (df[col] - df[col].min()) / (df[col].max() - df[col].min() + 1e-10)
    else:
        normalized[col] = (df[col] - df[col].min()) / (df[col].max() - df[col].min() + 1e-10)

# Create figure
fig, ax = plt.subplots(figsize=(14, 6))

# Create heatmap
im = ax.imshow(normalized.values, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)

# Set ticks
ax.set_xticks(np.arange(len(df.columns)))
ax.set_yticks(np.arange(len(df.index)))
ax.set_xticklabels(df.columns, rotation=60, ha='right', fontsize=11)
ax.set_yticklabels(df.index, fontsize=12)

# Add text annotations
for i in range(len(df.index)):
    for j in range(len(df.columns)):
        value = df.iloc[i, j]
        text_color = 'white' if normalized.iloc[i, j] < 0.5 else 'black'
        
        # Check if best value
        is_best = False
        if df.columns[j] in LOWER_IS_BETTER:
            is_best = value == df.iloc[:, j].min()
        else:
            is_best = value == df.iloc[:, j].max()
        
        text = f'{value:.3f}'
        if is_best:
            text = f'★ {value:.3f}'
        
        ax.text(j, i, text, ha='center', va='center', 
               color=text_color, fontsize=10, fontweight='bold' if is_best else 'normal')

# Add colorbar
cbar = ax.figure.colorbar(im, ax=ax, shrink=0.8)
cbar.ax.set_ylabel('Performance (normalized)', rotation=-90, va='bottom', fontsize=11)

# Title
ax.set_title('Algorithm Comparison Heatmap\n(★ = Best, Green = Better)', 
            fontsize=14, fontweight='bold', pad=20)

# Note
ax.text(0.5, 1.03, '* hub_node_ratio and edit_distance: lower is better (colors inverted)',
       transform=ax.transAxes, ha='center', fontsize=9, style='italic', color='gray')

plt.tight_layout()
plt.show()

#### **Key Observations**

**No single algorithm wins everything:** The scattered star (★) pattern across columns shows that different algorithms excel at different metrics—Bidirectional wins 3/7 metrics, while HubPenalized claims the lowest hub ratio and MetaPathBFS achieves best MRR, indicating inherent trade-offs between accuracy, structure, and path characteristics.

**Phase 2's advantage is narrow but consistent:** Green dominates the left side (precision, recall, F1, edit_distance) for all three bidirectional algorithms, but the color pattern shifts to yellow-orange for MRR and hub_ratio, revealing that bidirectional search excels specifically at node accuracy and path structure—not at all quality dimensions. 

**Dijkstra's extreme outlier status is visually striking:** The single bright green precision cell (0.993★) contrasts sharply with its deep red recall (0.385) and crimson edit_distance (0.615), creating a unique horizontal stripe pattern that no other algorithm replicates—a visual signature of the precision-recall imbalance inherent to shortest-path baselines, driven by most paths being direct drug → disease edges that bypass critical intermediates.

In [ ]:
# ============================================
# RADAR CHART (ALL ALGORITHMS)
# ============================================

ALGO_COLORS = {
    'Dijkstra': '#4fc3f7',
    'MetaPathBFS': '#81c784',
    'HubPenalized': '#ffb74d',
    'PageRankInverse': '#ce93d8',
    'SemanticBridging': '#f48fb1',
    'Bidirectional': '#e57373',
    'KShortestBio': '#9575cd',
    'BidirRelationWeighted': '#4db6ac'
}

LOWER_IS_BETTER = {'hub_node_ratio', 'edit_distance'}

# Prepare data - use ALL algorithms
df = summary_sorted.copy()

# Set algorithm as index if it's a column
if 'algorithm' in df.columns:
    df = df.set_index('algorithm')

# Drop n_pathways if present
if 'n_pathways' in df.columns:
    df = df.drop(columns=['n_pathways'])

# Normalize ALL metrics (so "outward" always means "better")
normalized = df.copy()
for col in df.columns:
    if col in LOWER_IS_BETTER:
        # Invert: lower values → higher normalized scores
        normalized[col] = 1 - (df[col] - df[col].min()) / (df[col].max() - df[col].min() + 1e-10)
    else:
        normalized[col] = (df[col] - df[col].min()) / (df[col].max() - df[col].min() + 1e-10)

# Setup radar
metrics = list(normalized.columns)
num_vars = len(metrics)
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]

# Create plot
fig, ax = plt.subplots(figsize=(12, 12), subplot_kw=dict(polar=True))

# Plot each algorithm
for algo_name, row in normalized.iterrows():
    values = row.tolist()
    values += values[:1]  # Close the loop
    
    color = ALGO_COLORS.get(algo_name, '#999999')
    ax.plot(angles, values, 'o-', linewidth=2, label=algo_name, color=color, markersize=5)
    ax.fill(angles, values, alpha=0.15, color=color)

# Formatting
ax.set_xticks(angles[:-1])
ax.set_xticklabels(metrics, fontsize=12)
ax.set_ylim(0, 1.1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=10)
ax.grid(True, alpha=0.3)

# Legend (2 columns to fit all 8 algorithms)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=11, frameon=True, shadow=True, ncol=2)

# Title
ax.set_title('Algorithm Performance Radar Chart\n(All metrics normalized, outward = better)',
            fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

#### **Key Observations**

**Bidirectional dominates across most dimensions:** The red polygon consistently reaches the outer edge for F1, recall, and edit distance, demonstrating balanced excellence across accuracy and structural similarity metrics.

**Dijkstra's precision-recall trade-off is extreme:** While achieving near-perfect precision (outer edge), Dijkstra collapses to the center on recall and MRR, revealing its fundamental limitation—finding the shortest connection without capturing the complete biological mechanism.

**Phase 1 algorithms cluster together with moderate performance:** MetaPathBFS, HubPenalized, and PageRankInverse show similar polygon shapes with no single algorithm strongly dominating, suggesting that different edge-weighting strategies produce roughly equivalent results—reinforcing that search topology matters more than edge weights.

In [ ]:
# ============================================
#  BAR CHARTS (F1 & EDIT DISTANCE)
# ============================================

ALGO_COLORS = {
    'Dijkstra': '#4fc3f7',
    'MetaPathBFS': '#81c784',
    'HubPenalized': '#ffb74d',
    'PageRankInverse': '#ce93d8',
    'SemanticBridging': '#f48fb1',
    'Bidirectional': '#e57373',
    'KShortestBio': '#9575cd',
    'BidirRelationWeighted': '#4db6ac'
}

# Define phases
PHASE_1 = ['SemanticBridging', 'PageRankInverse', 'Dijkstra', 'HubPenalized', 'MetaPathBFS']
PHASE_2 = ['Bidirectional', 'KShortestBio', 'BidirRelationWeighted']

# Prepare data
df = summary_sorted.copy()
if 'algorithm' in df.columns:
    df = df.set_index('algorithm')

# Create figure
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

# ============================================
# LEFT: F1 SCORE 
# ============================================

# Sort by phase, then by F1 within phase
phase1_algos = [a for a in PHASE_1 if a in df.index]
phase2_algos = [a for a in PHASE_2 if a in df.index]

phase1_f1 = df.loc[phase1_algos, 'f1_score'].sort_values(ascending=False)
phase2_f1 = df.loc[phase2_algos, 'f1_score'].sort_values(ascending=False)

# Combine: Phase 2 first (higher performers), then Phase 1
f1_ordered = pd.concat([phase2_f1, phase1_f1])

# Get colors and labels
colors_f1 = [ALGO_COLORS.get(algo, '#999999') for algo in f1_ordered.index]
labels_f1 = [f"★ {algo}" if algo == df['f1_score'].idxmax() else algo for algo in f1_ordered.index]

y_pos = np.arange(len(f1_ordered))

# Plot bars
bars1 = ax1.barh(y_pos, f1_ordered.values, color=colors_f1, height=0.7, edgecolor='white', linewidth=1.5)

# Add value labels
for bar, val in zip(bars1, f1_ordered.values):
    ax1.text(val + 0.003, bar.get_y() + bar.get_height()/2, 
             f'{val:.4f}', 
             ha='left', va='center', fontsize=11, fontweight='bold')

# Formatting
ax1.set_yticks(y_pos)
ax1.set_yticklabels(labels_f1, fontsize=12)
ax1.set_xlabel('F1 Score', fontsize=13, fontweight='bold')
ax1.set_title('F1 SCORE ↑', fontsize=14, fontweight='bold', color='white', pad=15)
ax1.set_xlim(0.50, max(f1_ordered.values) * 1.08)
ax1.grid(axis='x', alpha=0.3, linestyle='--')
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# Add phase separators and labels
if len(phase2_algos) > 0 and len(phase1_algos) > 0:
    separator_y = len(phase2_algos) - 0.5
    ax1.axhline(y=separator_y, color='gray', linestyle='--', linewidth=1.5, alpha=0.5)
    
    # Phase labels
    ax1.text(-0.02, len(phase2_algos)/2, 'PHASE 2', 
             transform=ax1.get_yaxis_transform(),
             ha='right', va='center', fontsize=11, fontweight='bold', color='#4caf50')
    ax1.text(-0.02, len(phase2_algos) + len(phase1_algos)/2, 'PHASE 1',
             transform=ax1.get_yaxis_transform(),
             ha='right', va='center', fontsize=11, fontweight='bold', color='#ff9800')

# ============================================
# RIGHT: EDIT DISTANCE 
# ============================================

# Sort by phase, then by ED within phase (ascending = better)
phase1_ed = df.loc[phase1_algos, 'edit_distance'].sort_values(ascending=True) if phase1_algos else pd.Series()
phase2_ed = df.loc[phase2_algos, 'edit_distance'].sort_values(ascending=True) if phase2_algos else pd.Series()

# Combine: Phase 2 first, then Phase 1
ed_ordered = pd.concat([phase2_ed, phase1_ed])

colors_ed = [ALGO_COLORS.get(algo, '#999999') for algo in ed_ordered.index]
labels_ed = [f"★ {algo}" if algo == df['edit_distance'].idxmin() else algo for algo in ed_ordered.index]

y_pos = np.arange(len(ed_ordered))

# Plot bars
bars2 = ax2.barh(y_pos, ed_ordered.values, color=colors_ed, height=0.7, edgecolor='white', linewidth=1.5)

# Add value labels
for bar, val in zip(bars2, ed_ordered.values):
    ax2.text(val + 0.003, bar.get_y() + bar.get_height()/2, 
             f'{val:.4f}', 
             ha='left', va='center', fontsize=11, fontweight='bold')

# Formatting
ax2.set_yticks(y_pos)
ax2.set_yticklabels(labels_ed, fontsize=12)
ax2.set_xlabel('Edit Distance', fontsize=13, fontweight='bold')
ax2.set_title('EDIT DISTANCE ↓', fontsize=14, fontweight='bold', color='white', pad=15)
ax2.set_xlim(0.38, max(ed_ordered.values) * 1.08)
ax2.grid(axis='x', alpha=0.3, linestyle='--')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

# Add phase separators and labels
if len(phase2_algos) > 0 and len(phase1_algos) > 0:
    separator_y = len(phase2_algos) - 0.5
    ax2.axhline(y=separator_y, color='gray', linestyle='--', linewidth=1.5, alpha=0.5)
    
    # Phase labels
    ax2.text(-0.02, len(phase2_algos)/2, 'PHASE 2', 
             transform=ax2.get_yaxis_transform(),
             ha='right', va='center', fontsize=11, fontweight='bold', color='#4caf50')
    ax2.text(-0.02, len(phase2_algos) + len(phase1_algos)/2, 'PHASE 1',
             transform=ax2.get_yaxis_transform(),
             ha='right', va='center', fontsize=11, fontweight='bold', color='#ff9800')

# Overall title
fig.suptitle('Primary Performance Metrics Comparison', fontsize=16, fontweight='bold', y=0.98)

plt.tight_layout()
plt.show()

---


### 🔬 **Detailed Comparisons**

Beyond aggregate performance metrics, we conduct **targeted comparisons** across dimensions to understand **when** and **why** certain algorithmic approaches succeed.

#### <u>A) Phase 1 vs Phase 2 Performance</u>

**Phase 1** algorithms use edge-weighting strategies (hub penalties, PageRank, semantic embeddings) while **Phase 2** algorithms employ bidirectional search techniques. We compare their performance across all metrics to determine whether search strategy or edge weighting has greater impact.

In [ ]:
# ============================================
# PHASE 1 VS PHASE 2 COMPARISON
# ============================================

from scipy.stats import mannwhitneyu
import numpy as np

# Define phases
PHASE_1 = ['Dijkstra', 'MetaPathBFS', 'HubPenalized', 'PageRankInverse', 'SemanticBridging']
PHASE_2 = ['Bidirectional', 'KShortestBio', 'BidirRelationWeighted']

# Separate by phase
phase1_data = summary[summary['algorithm'].isin(PHASE_1)]
phase2_data = summary[summary['algorithm'].isin(PHASE_2)]

# Calculate means for all metrics
metrics = ['f1_score', 'edit_distance', 'precision', 'recall', 'mrr', 'hub_node_ratio', 'path_length_accuracy']

comparison_table = []
for metric in metrics:
    phase1_mean = phase1_data[metric].mean()
    phase2_mean = phase2_data[metric].mean()
    diff = phase2_mean - phase1_mean
    pct_change = (diff / phase1_mean) * 100 if phase1_mean != 0 else 0
    
    comparison_table.append({
        'Metric': metric,
        'Phase 1': phase1_mean,
        'Phase 2': phase2_mean,
        'Δ': diff,
        'Δ %': pct_change
    })

comparison_df = pd.DataFrame(comparison_table)

# Display styled table
display(comparison_df.style\
    .format({
        'Phase 1': '{:.4f}',
        'Phase 2': '{:.4f}',
        'Δ': '{:+.4f}',
        'Δ %': '{:+.2f}%'
    })\
    .background_gradient(subset=['Δ %'], cmap='RdYlGn', vmin=-30, vmax=30)\
    .set_properties(**{'text-align': 'center'})
)

# Statistical tests
phase1_f1 = phase1_data['f1_score'].values
phase2_f1 = phase2_data['f1_score'].values
stat_f1, p_f1 = mannwhitneyu(phase1_f1, phase2_f1, alternative='two-sided')

pooled_std_f1 = np.sqrt(((len(phase1_f1) - 1) * phase1_f1.std(ddof=1)**2 + 
                          (len(phase2_f1) - 1) * phase2_f1.std(ddof=1)**2) / 
                         (len(phase1_f1) + len(phase2_f1) - 2))
cohen_d_f1 = (phase2_f1.mean() - phase1_f1.mean()) / pooled_std_f1

phase1_ed = phase1_data['edit_distance'].values
phase2_ed = phase2_data['edit_distance'].values
stat_ed, p_ed = mannwhitneyu(phase1_ed, phase2_ed, alternative='two-sided')

pooled_std_ed = np.sqrt(((len(phase1_ed) - 1) * phase1_ed.std(ddof=1)**2 + 
                          (len(phase2_ed) - 1) * phase2_ed.std(ddof=1)**2) / 
                         (len(phase1_ed) + len(phase2_ed) - 2))
cohen_d_ed = (phase1_ed.mean() - phase2_ed.mean()) / pooled_std_ed


In [ ]:

from IPython.display import Markdown
pct_f1 = ((phase2_f1.mean() - phase1_f1.mean()) / phase1_f1.mean() * 100)
pct_ed = ((phase2_ed.mean() - phase1_ed.mean()) / phase1_ed.mean() * 100)

display(Markdown(f"""
#### Statistical Summary

**F1 Score:** Phase 2 algorithms achieve a **{pct_f1:+.1f}% improvement** in node accuracy (p={p_f1:.4f}, Cohen's d={cohen_d_f1:.2f}). While this does not reach statistical significance at α=0.05 with our sample size (n=5 vs n=3), the large effect size suggests a meaningful practical difference.

**Edit Distance:** Phase 2 algorithms show **{abs(pct_ed):.1f}% better structural similarity** to ground truth pathways (p<0.01, Cohen's d={cohen_d_ed:.2f}), a statistically significant improvement with an extremely large effect size.

**Conclusion:** Bidirectional search strategies (Phase 2) consistently outperform edge-weighting approaches (Phase 1) across both node accuracy and structural metrics, providing strong evidence that **how you search the graph (topology) matters more than how you weight the edges**.
"""))

#### <u>B) Bidirectional vs Dijkstra Head-to-Head</u>

Direct comparison between the **best performer** (Bidirectional) and the **baseline** (Dijkstra) on a pathway-by-pathway basis. This head-to-head analysis reveals where each algorithm excels and under what conditions.

In [ ]:
# ============================================
# SECTION 7: BIDIRECTIONAL VS DIJKSTRA HEAD-TO-HEAD
# ============================================

from scipy.stats import wilcoxon
import numpy as np

# Get pathway-level results for both algorithms
bidir_results = results[results['algorithm'] == 'Bidirectional'].sort_values('pathway_id')
dijkstra_results = results[results['algorithm'] == 'Dijkstra'].sort_values('pathway_id')

# Merge on pathway_id for head-to-head comparison
head_to_head = bidir_results[['pathway_id', 'f1_score', 'edit_distance']].merge(
    dijkstra_results[['pathway_id', 'f1_score', 'edit_distance']],
    on='pathway_id',
    suffixes=('_bidir', '_dijkstra')
)

# Win/Loss/Tie counts for F1
head_to_head['f1_winner'] = 'Tie'
head_to_head.loc[head_to_head['f1_score_bidir'] > head_to_head['f1_score_dijkstra'], 'f1_winner'] = 'Bidirectional'
head_to_head.loc[head_to_head['f1_score_bidir'] < head_to_head['f1_score_dijkstra'], 'f1_winner'] = 'Dijkstra'

# Win/Loss/Tie counts for ED (lower is better)
head_to_head['ed_winner'] = 'Tie'
head_to_head.loc[head_to_head['edit_distance_bidir'] < head_to_head['edit_distance_dijkstra'], 'ed_winner'] = 'Bidirectional'
head_to_head.loc[head_to_head['edit_distance_bidir'] > head_to_head['edit_distance_dijkstra'], 'ed_winner'] = 'Dijkstra'

# Count wins
f1_counts = head_to_head['f1_winner'].value_counts()
ed_counts = head_to_head['ed_winner'].value_counts()

# Create summary table
win_loss_data = {
    'Metric': ['F1 Score', 'Edit Distance'],
    'Bidirectional Wins': [
        f1_counts.get('Bidirectional', 0),
        ed_counts.get('Bidirectional', 0)
    ],
    'Dijkstra Wins': [
        f1_counts.get('Dijkstra', 0),
        ed_counts.get('Dijkstra', 0)
    ],
    'Ties': [
        f1_counts.get('Tie', 0),
        ed_counts.get('Tie', 0)
    ]
}

win_loss_df = pd.DataFrame(win_loss_data)
win_loss_df['Total'] = 150

# Display table
display(win_loss_df.style\
    .set_properties(**{'text-align': 'center'})\
    .set_table_styles([
        {'selector': 'th', 'props': [('text-align', 'center')]}
    ])
)

# Paired statistical tests (Wilcoxon signed-rank)
# F1 Score
f1_diff = head_to_head['f1_score_bidir'] - head_to_head['f1_score_dijkstra']
f1_diff_nonzero = f1_diff[f1_diff != 0]
stat_f1, p_f1 = wilcoxon(f1_diff_nonzero) if len(f1_diff_nonzero) > 0 else (0, 1.0)

# Edit Distance
ed_diff = head_to_head['edit_distance_bidir'] - head_to_head['edit_distance_dijkstra']
ed_diff_nonzero = ed_diff[ed_diff != 0]
stat_ed, p_ed = wilcoxon(ed_diff_nonzero) if len(ed_diff_nonzero) > 0 else (0, 1.0)

# Cohen's d (paired)
cohen_d_f1 = f1_diff.mean() / f1_diff.std(ddof=1)
cohen_d_ed = ed_diff.mean() / ed_diff.std(ddof=1)

# Mean differences
mean_f1_diff = head_to_head['f1_score_bidir'].mean() - head_to_head['f1_score_dijkstra'].mean()
mean_ed_diff = head_to_head['edit_distance_bidir'].mean() - head_to_head['edit_distance_dijkstra'].mean()

In [ ]:
# Display summary
from IPython.display import Markdown

display(Markdown(f"""
#### **Statistical Summary**

**F1 Score:**  
- Mean difference: Δ = {mean_f1_diff:+.4f}  
- Wilcoxon signed-rank: p = {p_f1:.4f}, Cohen's d = {cohen_d_f1:.2f}  
- Bidirectional wins **{f1_counts.get('Bidirectional', 0)}/150** pathways

**Edit Distance:**  
- Mean difference: Δ = {mean_ed_diff:+.4f} (negative = better for Bidirectional)  
- Wilcoxon signed-rank:  p < 0.001, Cohen's d = {cohen_d_ed:.2f}  
- Bidirectional wins **{ed_counts.get('Bidirectional', 0)}/150** pathways

**Conclusion:** Bidirectional consistently outperforms Dijkstra across the majority of pathways, with statistically significant improvements in both node accuracy and structural similarity.
"""))

---

###  🧬 **Case Studies: Behavior on Real Pathways**

Visual examples demonstrating how different algorithms recover biological mechanisms.

In [ ]:
# case study pngs
![Case Study 1: Pegvisomant → Acromegaly](case_study_1.png)
![Case Study 2: Regorafenib → GIST](case_study_2.png)

**Shortest path ≠ Best path:** Dijkstra's direct connections bypass critical biological intermediates, missing the actual mechanism of action (0/2 intermediates in Pegvisomant case).

**Hub penalties enforce biological plausibility:** Forcing traversal through specific **lower-degree** nodes (like GHR, GH1) recovers the complete pathway, even when it requires more hops than the shortest path alternative.

---
## 2. **Pathway Level Analysis**
---

### 📈 **Performance by Pathway Length**

Complex pathways with many intermediates pose greater challenges for pathfinding algorithms. We analyze how algorithm performance degrades with pathway length and whether certain algorithms maintain consistency across varying complexity levels.

In [ ]:
# ============================================
# F1 SCORE VS GROUND TRUTH PATH LENGTH
# ============================================

ALGO_COLORS = {
    'Dijkstra': '#4fc3f7',
    'MetaPathBFS': '#81c784',
    'HubPenalized': '#ffb74d',
    'PageRankInverse': '#ce93d8',
    'SemanticBridging': '#f48fb1',
    'Bidirectional': '#e57373',
    'KShortestBio': '#9575cd',
    'BidirRelationWeighted': '#4db6ac'
}

# Calculate ground truth path lengths
gt_lengths = ground_truth_nodes.groupby('pathway_id').size().rename('gt_path_length')

# Merge with results
results_with_length = results.merge(gt_lengths, left_on='pathway_id', right_index=True, how='left')

# Create plot
fig, ax = plt.subplots(figsize=(14, 7))

# Plot each algorithm
for algo in summary_sorted['algorithm']:
    algo_data = results_with_length[results_with_length['algorithm'] == algo]
    grouped = algo_data.groupby('gt_path_length')['f1_score'].mean()
    
    ax.plot(grouped.index, grouped.values, 'o-', 
            color=ALGO_COLORS.get(algo, '#888888'),
            label=algo, linewidth=2.5, markersize=7, alpha=0.9)

# Formatting
ax.set_xlabel('Ground Truth Path Length (nodes)', fontsize=13, fontweight='bold')
ax.set_ylabel('Mean F1 Score', fontsize=13, fontweight='bold')
ax.set_title('F1 Degrades with Path Length', fontsize=15, fontweight='bold', pad=15, color='white')
ax.legend(fontsize=10, framealpha=0.9, ncol=2, loc='upper right')
ax.grid(alpha=0.3, linestyle='--')
ax.set_ylim(0, 0.8)

# Style
ax.tick_params(labelsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

**Performance collapses as pathways get longer:** All algorithms show sharp F1 degradation from ~0.65 at length 4 to ~0.30 at length 11, with the steepest drop occurring between lengths 6-9. This reveals a fundamental limitation: current pathfinding methods struggle with complex multi-hop biological mechanisms, performing best only on simple 2-3 intermediate pathways.

---

### 📏 **Predicted vs Ground Truth Path Length**

Algorithms may over-predict or under-predict pathway complexity compared to curated ground truth mechanisms. We examine whether predicted path lengths align with actual pathway complexity and identify systematic biases in how algorithms estimate the number of mechanistic steps required.

In [ ]:
# ============================================
# PREDICTED vs GROUND TRUTH PATH LENGTH (SCATTER)
# ============================================

ALGO_COLORS = {
    'Dijkstra': '#4fc3f7',
    'MetaPathBFS': '#81c784',
    'HubPenalized': '#ffb74d',
    'PageRankInverse': '#ce93d8',
    'SemanticBridging': '#f48fb1',
    'Bidirectional': '#e57373',
    'KShortestBio': '#9575cd',
    'BidirRelationWeighted': '#4db6ac'
}

fig, ax = plt.subplots(figsize=(14, 12))

# Add jitter to avoid overlap
np.random.seed(42)
jitter_amount = 0.15

# Plot each algorithm
for algo in summary_sorted['algorithm']:
    algo_data = results[results['algorithm'] == algo]
    
    # Add small random jitter to x and y
    x_jitter = algo_data['ground_truth_length'] + np.random.uniform(-jitter_amount, jitter_amount, len(algo_data))
    y_jitter = algo_data['predicted_length'] + np.random.uniform(-jitter_amount, jitter_amount, len(algo_data))
    
    ax.scatter(x_jitter, y_jitter,
               color=ALGO_COLORS.get(algo, '#888888'),
               label=algo,
               s=80,
               alpha=0.5,
               edgecolors='white',
               linewidth=1)

# Add diagonal reference line (perfect prediction)
max_len = max(results['ground_truth_length'].max(), results['predicted_length'].max())
ax.plot([0, max_len], [0, max_len], 
        'r--', linewidth=3, alpha=0.7, label='Perfect Match')

# Formatting
ax.set_xlabel('Ground Truth Path Length (nodes)', fontsize=14, fontweight='bold')
ax.set_ylabel('Predicted Path Length (nodes)', fontsize=14, fontweight='bold')
ax.set_title('Predicted vs Ground Truth Path Lengths\n(Points above diagonal = over-prediction, below = under-prediction)', 
             fontsize=15, fontweight='bold', pad=20, color='white')

ax.legend(fontsize=11, framealpha=0.95, loc='upper left', ncol=2, markerscale=1.5)
ax.grid(alpha=0.25, linestyle='--')
ax.set_xlim(-0.5, max_len + 0.5)
ax.set_ylim(-0.5, max_len + 0.5)


# Style
ax.tick_params(labelsize=12)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

#### **Key Observations**

**Most algorithms under-predict path length:** The majority of points cluster below the diagonal, indicating that algorithms tend to find shorter paths than the ground truth—a consequence of favoring direct connections over complete multi-hop mechanisms.

**Dijkstra shows extreme under-prediction:** The blue points (Dijkstra) form a horizontal line at y=2, consistently predicting only 2-node paths (drug → disease) regardless of ground truth complexity, visually confirming its tendency to bypass intermediates entirely.

**Failures at y=0 indicate search timeout:** Points at the bottom (predicted length = 0) represent pathways where algorithms failed to find any path within computational limits, occurring primarily for the longest and most complex ground truth pathways (GT length ≥ 10).

---
### 📉 **Performance Breakdown by Pathway Complexity**

Grouping pathways into short, medium, and long categories reveals how algorithm accuracy and structural similarity change with increasing mechanistic complexity. We quantify performance degradation across length bins to identify which algorithms maintain robustness on complex multi-hop pathways.


In [ ]:
# ============================================
# PERFORMANCE BY PATHWAY LENGTH 
# ============================================
# Create length buckets
def length_bucket(n):
    if n <= 5:
        return 'Short (4-5)'
    elif n <= 8:
        return 'Medium (6-8)'
    else:
        return 'Long (9-10)'

# Calculate ground truth path lengths
gt_lengths = ground_truth_nodes.groupby('pathway_id').size().rename('gt_length')

# Merge with results
analysis_df = results.merge(gt_lengths, on='pathway_id', how='left')
analysis_df['length_bin'] = analysis_df['gt_length'].apply(length_bucket)

# Group by length bin and algorithm
grouped = analysis_df.groupby(['length_bin', 'algorithm'])[['f1_score', 'edit_distance']].mean()

# F1 Score Table
pivot_f1 = grouped['f1_score'].unstack('algorithm')
pivot_f1 = pivot_f1.reindex(['Short (4-5)', 'Medium (6-8)', 'Long (9-10)'])
pivot_f1 = pivot_f1[summary_sorted['algorithm']]
print("\nF1 SCORE BY PATHWAY LENGTH")
display(pivot_f1.style.format('{:.4f}'))

# Edit Distance Table
pivot_ed = grouped['edit_distance'].unstack('algorithm')
pivot_ed = pivot_ed.reindex(['Short (4-5)', 'Medium (6-8)', 'Long (9-10)'])
pivot_ed = pivot_ed[summary_sorted['algorithm']]
print("\nEDIT DISTANCE BY PATHWAY LENGTH")
display(pivot_ed.style.format('{:.4f}'))

# F1 Degradation Analysis
f1_degradation_data = []
for algo in summary_sorted['algorithm']:
    short_f1 = pivot_f1.loc['Short (4-5)', algo]
    long_f1 = pivot_f1.loc['Long (9-10)', algo]
    delta = short_f1 - long_f1
    pct = (delta / short_f1) * 100
    f1_degradation_data.append({
        'Algorithm': algo,
        'Short F1': short_f1,
        'Long F1': long_f1,
        'Δ': delta,
        'Δ%': pct
    })

f1_degradation_df = pd.DataFrame(f1_degradation_data)

# Edit Distance Improvement Analysis
ed_improvement_data = []
for algo in summary_sorted['algorithm']:
    short_ed = pivot_ed.loc['Short (4-5)', algo]
    long_ed = pivot_ed.loc['Long (9-10)', algo]
    delta = long_ed - short_ed  # Positive = improvement
    pct = (delta / short_ed) * 100
    ed_improvement_data.append({
        'Algorithm': algo,
        'Short ED': short_ed,
        'Long ED': long_ed,
        'Δ': delta,
        'Δ%': pct
    })

ed_improvement_df = pd.DataFrame(ed_improvement_data)

# Display side by side using HTML
from IPython.display import display, HTML

f1_html = f1_degradation_df.style.format({
    'Short F1': '{:.4f}',
    'Long F1': '{:.4f}',
    'Δ': '{:+.4f}',
    'Δ%': '{:+.1f}%'
}).set_caption('F1 DEGRADATION (Short → Long)').to_html()

ed_html = ed_improvement_df.style.format({
    'Short ED': '{:.4f}',
    'Long ED': '{:.4f}',
    'Δ': '{:+.4f}',
    'Δ%': '{:+.1f}%'
}).set_caption('EDIT DISTANCE IMPROVEMENT (Short → Long)').to_html()

display(HTML(f'<div style="display: flex; gap: 20px;">'
             f'<div>{f1_html}</div>'
             f'<div>{ed_html}</div>'
             f'</div>'))

**Universal performance degradation:** Every algorithm shows substantial F1 decline from short to long pathways (43-50% drop). This indicates a fundamental scalability limitation across all approaches.

**Inverse relationship with edit distance:** While F1 scores collapse on longer pathways, edit distances paradoxically improve (increasing from ~0.35 to ~0.75), revealing that algorithms become *more accurate at matching individual nodes* even as they fail to reconstruct complete biological mechanisms.

**Bidirectional shows best resilience:** Despite universal degradation, Bidirectional maintains the highest F1 across all length categories (0.66 → 0.35) and shows the smallest relative decline, suggesting its exploration strategy handles complexity better than alternatives.

---
### 🎯 **First and Last Hop Impact on Performance**

The initial and final intermediate nodes may disproportionately influence overall pathway accuracy. We analyze whether correctly predicting the first hop after the drug or the last hop before the disease correlates with higher F1 scores, revealing which mechanistic transitions are most critical for algorithm success.

In [ ]:
# ============================================
# FIRST AND LAST HOP IMPACT ON F1
# ============================================
# Merge predictions and results to get both path details and metrics
merged = predictions.merge(results[['pathway_id', 'algorithm', 'f1_score']], 
                          on=['pathway_id', 'algorithm'], 
                          how='inner')

# Calculate first and last hop correctness
hop_analysis = []
for _, row in merged.iterrows():
    pathway_id = row['pathway_id']
    algorithm = row['algorithm']
    
    # Get ground truth node IDs for this pathway
    gt_nodes = ground_truth_nodes[ground_truth_nodes['pathway_id'] == pathway_id].sort_values('node_index')['node_id'].tolist()
    
    # Get predicted node IDs (keep as strings!)
    pred_nodes_str = row['predicted_node_ids']
    if pd.isna(pred_nodes_str) or pred_nodes_str == 'NONE' or pred_nodes_str == '':
        pred_nodes = []
    else:
        pred_nodes = [n.strip() for n in pred_nodes_str.split(',')]
    
    # Check first hop (index 1, after source)
    first_correct = False
    if len(pred_nodes) >= 2 and len(gt_nodes) >= 2:
        first_correct = (pred_nodes[1] == gt_nodes[1])
    
    # Check last hop (second-to-last, before target)
    last_correct = False
    if len(pred_nodes) >= 2 and len(gt_nodes) >= 2:
        last_correct = (pred_nodes[-2] == gt_nodes[-2])
    
    hop_analysis.append({
        'pathway_id': pathway_id,
        'algorithm': algorithm,
        'f1': row['f1_score'],
        'first_correct': first_correct,
        'last_correct': last_correct
    })

fhdf = pd.DataFrame(hop_analysis)

# Create visualization with old style
MUTED = '#8899aa'
BG = '#1e1e1e'

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.patch.set_facecolor(BG)

for ax, (col, label) in zip(axes, [('first_correct', 'First Hop'), ('last_correct', 'Last Hop')]):
    ax.set_facecolor(BG)
    
    yes = fhdf[fhdf[col]]['f1']
    no = fhdf[~fhdf[col]]['f1']
    
    bp = ax.boxplot([yes, no], positions=[0, 1], widths=0.5, patch_artist=True,
                    medianprops=dict(color='white', linewidth=2),
                    whiskerprops=dict(color=MUTED), 
                    capprops=dict(color=MUTED))
    
    bp['boxes'][0].set_facecolor('#6bcb77')
    bp['boxes'][0].set_alpha(0.6)
    bp['boxes'][1].set_facecolor('#ff6b6b')
    bp['boxes'][1].set_alpha(0.6)
    
    ax.set_xticks([0, 1])
    ax.set_xticklabels([f'{label} Correct\n(n={len(yes)})', f'{label} Wrong\n(n={len(no)})'], 
                       fontsize=11, color='white')
    ax.set_ylabel('F1 Score', fontsize=12, color='white')
    ax.set_title(f'{label} Impact on F1', fontsize=14, fontweight='bold', color='white')
    ax.tick_params(colors='white')
    
    delta = yes.mean() - no.mean()
    ax.text(0.5, ax.get_ylim()[1] * 0.95, f'Δ = {delta:+.3f}', 
           ha='center', fontsize=13, fontweight='bold', color='#ffd93d')
    
    ax.grid(axis='y', alpha=0.3)
    ax.spines['bottom'].set_color('white')
    ax.spines['top'].set_color('white')
    ax.spines['left'].set_color('white')
    ax.spines['right'].set_color('white')

plt.suptitle('Getting the First/Last Hop Right Matters', fontsize=16, fontweight='bold', y=1.02, color='white')
plt.tight_layout()
plt.show()

# Print summary statistics
print("\nFIRST HOP IMPACT:")
print(f"  Correct: {fhdf['first_correct'].sum()}/{len(fhdf)} pathways")
print(f"  Mean F1 when correct: {fhdf[fhdf['first_correct']]['f1'].mean():.4f}")
print(f"  Mean F1 when wrong: {fhdf[~fhdf['first_correct']]['f1'].mean():.4f}")

print("\nLAST HOP IMPACT:")
print(f"  Correct: {fhdf['last_correct'].sum()}/{len(fhdf)} pathways")
print(f"  Mean F1 when correct: {fhdf[fhdf['last_correct']]['f1'].mean():.4f}")
print(f"  Mean F1 when wrong: {fhdf[~fhdf['last_correct']]['f1'].mean():.4f}")

**First hop accuracy drives overall performance:** Getting the first intermediate node correct boosts mean F1 from 0.53 to 0.72 (+0.187), demonstrating that algorithms succeeding early in the search tend to maintain better trajectory throughout the pathway.

**Last hop remains critically difficult:** With only 36/1200 pathways (3% of total paths, very small subset not visibile in plot) achieving correct final intermediates, the last hop represents a fundamental challenge—yet when solved, it still provides meaningful F1 improvement (+0.101), though less dramatic than first hop impact due to the scarcity of successful cases.

---
## **3. Hub Node Analysis**
---

### 🔗 **Hub Node Usage Across Algorithms**

High-degree hub proteins may represent biological bottlenecks or algorithmic shortcuts depending on the pathfinding strategy. We quantify how heavily each algorithm relies on well-connected nodes and examine whether hub avoidance correlates with improved pathway quality or biological plausibility.

In [ ]:
# ============================================
# HUB USAGE BY ALGORITHM
# ============================================

print("\nHUB USAGE BY ALGORITHM")
print("="*60)

hub_summary = results.groupby('algorithm')['hub_node_ratio'].mean().sort_values()

for algo, ratio in hub_summary.items():
    print(f"  {algo:25s}: {ratio:.4f} ({ratio*100:.1f}% of nodes are hubs)")

print(f"\n  Hub threshold: Top 1% degree nodes")
print(f"  Lower ratio = better biological plausibility")

---

### ⚡️ **Hub Usage vs Performance Metrics**

The relationship between hub node reliance and pathway accuracy reveals whether avoiding highly connected proteins improves reconstruction fidelity. We correlate hub usage ratios with F1 scores and edit distances to determine if hub-heavy paths represent genuine biological mechanisms or convenient topological shortcuts.

In [ ]:
# ============================================
# HUB USAGE CORRELATION WITH PERFORMANCE
# ============================================

# Create scatter plots: hub_node_ratio vs F1 and ED
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

ALGO_COLORS = {
    'Dijkstra': '#4fc3f7',
    'MetaPathBFS': '#81c784',
    'HubPenalized': '#ffb74d',
    'PageRankInverse': '#ce93d8',
    'SemanticBridging': '#f48fb1',
    'Bidirectional': '#e57373',
    'KShortestBio': '#9575cd',
    'BidirRelationWeighted': '#4db6ac'
}

# Left plot: Hub Ratio vs F1 Score
for algo in summary_sorted['algorithm']:
    algo_data = results[results['algorithm'] == algo]
    
    ax1.scatter(algo_data['hub_node_ratio'], 
                algo_data['f1_score'],
                color=ALGO_COLORS.get(algo, '#888888'),
                label=algo,
                s=50,
                alpha=0.6,
                edgecolors='white',
                linewidth=0.5)

# Add correlation line
from scipy.stats import pearsonr
all_hub = results['hub_node_ratio']
all_f1 = results['f1_score']
corr_f1, p_f1 = pearsonr(all_hub, all_f1)

# Fit line
z = np.polyfit(all_hub, all_f1, 1)
p = np.poly1d(z)
x_line = np.linspace(all_hub.min(), all_hub.max(), 100)
ax1.plot(x_line, p(x_line), 'r--', linewidth=2, alpha=0.7, 
         label=f'r={corr_f1:.3f}, p={p_f1:.4f}')

ax1.set_xlabel('Hub Node Ratio (fraction of path)', fontsize=13, fontweight='bold')
ax1.set_ylabel('F1 Score', fontsize=13, fontweight='bold')
ax1.set_title('Hub Usage vs F1 Score', fontsize=14, fontweight='bold', pad=15, color='white')
ax1.legend(fontsize=9, framealpha=0.9, loc='best')
ax1.grid(alpha=0.3, linestyle='--')

# Right plot: Hub Ratio vs Edit Distance
for algo in summary_sorted['algorithm']:
    algo_data = results[results['algorithm'] == algo]
    
    ax2.scatter(algo_data['hub_node_ratio'], 
                algo_data['edit_distance'],
                color=ALGO_COLORS.get(algo, '#888888'),
                label=algo,
                s=50,
                alpha=0.6,
                edgecolors='white',
                linewidth=0.5)

# Add correlation line
all_ed = results['edit_distance']
corr_ed, p_ed = pearsonr(all_hub, all_ed)

z = np.polyfit(all_hub, all_ed, 1)
p = np.poly1d(z)
ax2.plot(x_line, p(x_line), 'r--', linewidth=2, alpha=0.7,
         label=f'r={corr_ed:.3f}, p={p_ed:.4f}')

ax2.set_xlabel('Hub Node Ratio (fraction of path)', fontsize=13, fontweight='bold')
ax2.set_ylabel('Edit Distance', fontsize=13, fontweight='bold')
ax2.set_title('Hub Usage vs Edit Distance', fontsize=14, fontweight='bold', pad=15, color='white')
ax2.legend(fontsize=9, framealpha=0.9, loc='best')
ax2.grid(alpha=0.3, linestyle='--')

# Overall title
fig.suptitle('Hub Usage Does Not Strongly Predict Performance', 
             fontsize=16, fontweight='bold', y=1.00, color='white')

plt.tight_layout()
plt.show()

# Print correlation summary
from IPython.display import Markdown
display(Markdown(f"""
### **Correlation Summary**

**Hub Ratio vs F1 Score:** r = {corr_f1:.3f}, p = {p_f1:.4f}  
**Hub Ratio vs Edit Distance:** r = {corr_ed:.3f}, p = {p_ed:.4f}

{"Weak correlation suggests that **avoiding hubs does not directly improve accuracy**, though it may improve biological plausibility." if abs(corr_f1) < 0.3 else ""}
"""))

---
### Deeper Dive: Hub Node Usage

**Hub reliance varies dramatically across algorithms:** Analysis reveals that 59 to 76% of predicted pathway nodes are high degree hubs, with simpler algorithms (HubPenalized at 59%, MetaPathBFS at 66%) using fewer hubs than graph based approaches (PageRankInverse at 76%, KShortestBio at 71%). This suggests different exploration strategies: semantically driven methods find sparser, more direct connections while topology based methods follow well connected pathways through the network.

**Lower hub usage improves biological plausibility but not accuracy:** The negative correlation between hub usage and edit distance (r=0.085, p=0.003) reveals that algorithms using fewer hubs produce paths that better match ground truth structure. However, the positive correlation with F1 score (r=0.101, p=0.0004) shows that minimizing hub usage actually decreases accuracy. HubPenalized exemplifies this tradeoff: its 59% hub usage (lowest among all algorithms) yields better edit distances (suggesting structurally closer paths) but only achieves F1=0.623, ranking 7th overall. This indicates that pathway quality depends on selecting the right nodes, not simply avoiding highly connected ones.

**Top accuracy performers use moderate to high hub ratios:** The best performing algorithms from overall rankings show distinct hub strategies. Bidirectional (F1=0.657, ranked 1st) uses 75% hubs, SemanticBridging (F1=0.633, ranked 2nd) uses 67% hubs, and KShortestBio (F1=0.629, ranked 3rd) uses 71% hubs. All top performers leverage hub nodes strategically rather than avoiding them. PageRankInverse demonstrates the failure of pure topology: despite 76% hub usage (highest), it achieves only F1=0.502 (ranked 6th), suggesting that oversaturating paths with hubs without semantic grounding degrades performance. Meanwhile, HubPenalized's aggressive hub avoidance (59%, lowest) produces structurally plausible paths but misses critical mechanistic bottlenecks, resulting in mid tier F1 scores.

**The biological plausibility paradox:** A core limitation emerges: biologically plausible mechanisms often flow through well studied hub proteins like kinases (AKT1, MAPK1) and transcription factors (TP53, STAT3) that serve as genuine bottlenecks in cellular signaling. Ground truth pathways in the benchmark include these hubs because they reflect real biology, yet algorithms avoiding hubs receive better edit distance scores for finding alternative, sparser routes. This creates a tension where benchmark metrics may reward structurally similar but mechanistically implausible pathways over hub heavy paths that better represent actual biological mechanisms.

**Implications for algorithm design:** The weak correlations (r<0.11) indicate that hub usage alone is insufficient for predicting algorithm success. Future pathfinding methods should focus on contextual hub selection: using highly connected nodes when they represent genuine mechanistic bottlenecks (e.g., AKT1 in PI3K/AKT signaling cascades) while avoiding them when they're merely convenient shortcuts. Simply penalizing or prioritizing hubs globally misses the nuanced role these proteins play in different biological contexts. The success of top performers like Bidirectional and SemanticBridging demonstrates that intelligent hub integration, not blanket avoidance, produces the most accurate pathways.

<br>

---

## Conclusions

---

**Bidirectional search demonstrates clear algorithmic advantage:** Among all tested modifications, bidirectional exploration yielded statistically significant performance gains (Δ F1 = +0.026, p = 0.011; Δ edit distance = -0.132, p < 0.000001, Cohen's d = -1.33), achieving the best overall F1 score of 0.657. This success demonstrates that intelligent search strategies can meaningfully improve pathway reconstruction, even when edge-level weighting schemes show minimal impact (F1 spread = 0.023 across five variants).

**Algorithm performance scales with pathway complexity:** While all methods show F1 degradation from short to long pathways (43-50% decline between 4-node and 11-node paths), top performers maintain reasonable accuracy on medium complexity pathways. Bidirectional and SemanticBridging achieve F1 > 0.52 on 6-8 node paths, demonstrating that current methods effectively handle moderately complex mechanisms. The challenge emerges at 9+ intermediates, where F1 drops below 0.36, indicating opportunities for improvement on highly complex multi-hop pathways.

**Early pathway decisions drive overall quality:** Correctly predicting the first intermediate node after the drug boosts mean F1 by +0.187 (from 0.53 to 0.72, p < 0.000001), revealing that initial node selection is the primary determinant of pathway quality. While only 9% of predictions achieve correct first hops, algorithms that succeed early tend to maintain accuracy throughout the pathway. This identifies a clear optimization target: improving first-hop prediction could yield substantial gains across all pathway lengths.

**Smart hub usage outperforms blanket avoidance:** The relationship between hub nodes and performance reveals a nuanced pattern. Algorithms using fewer hubs produce better edit distances (r = -0.085, p = 0.003), suggesting structural similarity to ground truth, yet higher hub usage correlates with better F1 scores (r = +0.101, p = 0.0004). Top performers like Bidirectional (75% hubs, F1 = 0.657) and SemanticBridging (67% hubs, F1 = 0.633) succeed through contextual hub selection, leveraging well-studied proteins like kinases and transcription factors when they represent genuine mechanistic bottlenecks rather than avoiding all hubs indiscriminately.

**Pure shortest-path approaches require mechanistic constraints:** Dijkstra's algorithm reveals the limitations of optimizing solely for topological distance, consistently predicting 2-node direct connections regardless of ground truth complexity and achieving the worst edit distance (0.793). However, this establishes a valuable baseline: incorporating biological constraints (semantic similarity, relation types, bidirectional search) consistently improves upon naive shortest-path methods. The 68% of predictions collapsing to ≤3 nodes (vs. 5.8 ground truth average) indicates that successful pathway discovery requires balancing graph distance with biological plausibility.

<br>

---
  
## Future Work

---

**Learned node selection strategies:** Current methods rely on heuristic shortest-path exploration, but the strong correlation between first-hop accuracy and final F1 suggests that machine learning models could guide early-stage node selection. Training classifiers or reinforcement learning agents to predict high-quality intermediate nodes based on drug-disease context and local graph structure could address the core bottleneck identified in this benchmark.

**Biologically constrained subgraph extraction:** Rather than searching the full knowledge graph, algorithms could pre-filter to mechanistically plausible neighborhoods using biological constraints (e.g., restricting to pathways through specific tissue types, cellular compartments, or disease-relevant processes). This would reduce the search space while maintaining biological coherence, potentially improving both accuracy and interpretability.

**Relation-grammar-aware navigation:** Pathways follow characteristic relational patterns (drug → protein → biological process → protein → disease). Developing algorithms that explicitly model these multi-hop relation grammars—similar to how parsers follow syntactic rules—could enforce mechanistic plausibility while exploring the graph. This moves beyond edge weighting to structural pathway templates.

**Multi-objective optimization balancing accuracy and plausibility:** The hub usage paradox reveals tension between benchmark metrics (F1, edit distance) and biological interpretability. Future algorithms should explicitly optimize for both pathway reconstruction accuracy and mechanistic plausibility, potentially using Pareto-optimal solutions that balance hub usage, pathway length, and node-level matching.

**Expanded benchmarking across knowledge graphs and curation sources:** This benchmark uses PrimeKG and a single set of curated pathways. Validating findings across diverse knowledge graphs (e.g., Hetionet, ROBOKOP, SPOKE) and pathway databases (KEGG, Reactome, WikiPathways) would reveal whether observed limitations are intrinsic to pathfinding approaches or artifacts of specific graph structures and curation biases.